In [26]:
import pandas as pd
import glob
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import openai
from bertopic.representation import OpenAI

from hdbscan import HDBSCAN

In [27]:
df_news = pd.read_parquet('./Data/all_news_1984_2024.parquet')
df_news['Date'] = pd.to_datetime(df_news['Date'], format='%Y-%m-%d')

In [10]:
year = 2001

df_gpr = pd.read_csv(f'./Result/GPR_Peaks/data/gpr_peaks_{year}.csv')
filtered_news = df_news[df_news['Date'].isin(df_gpr['date'])]
filtered_news

,Date,Caption,Content
887629,2001-09-12,Mistakes Made the Catastrophe Possible,It is likely that more Americans died yesterda...
887630,2001-09-12,How Did They Get Past Security?,"Now we know, someone said yesterday, why they ..."
887631,2001-09-12,Business World: What I Saw on the First Day of...,It's not every day in Manhattan that you look ...
887632,2001-09-12,"We Beat Hitler. We Can Vanquish This Foe, Too","America, it is said, is slow to awaken, and in..."
887633,2001-09-12,Civility Amid Chaos,Terrorists may have thought they were striking...
...,...,...,...
892574,2001-10-31,Argentine Agony,Argentine securities have hit the skids of lat...
892575,2001-10-31,Asides: Public Power Flickers,California's recent electricity crisis has rev...
892576,2001-10-31,The Case for Ground Troops,"During the Clinton years, the preferred milita..."
892577,2001-10-31,The Education of Jim McGreevey,"The way Jim McGreevey tells it, his childhood ..."


In [ ]:
# GICS_SEED_KEYWORDS = [
#     # 能源 (Energy)
#     ["energy", "oil", "gas", "opec", "shale", "chevron", "exxon", "natural gas", "pipeline"],
#     # 金融 (Financials)
#     ["financial", "bank", "fed", "ecb", "inflation", "stock", "equity", "bond", "investment", "goldman", "jpmorgan", "debt", "rates"],
#     # 信息技术 (Information Technology)
#     ["tech", "apple", "google", "microsoft", "semiconductor", "chip", "software", "ai", "nvidia", "meta"],
#     # 工业 (Industrials)
#     ["industrial", "boeing", "lockheed", "defense", "aerospace", "airline", "manufacturing", "supply chain"],
#     # 医疗保健 (Health Care)
#     ["health", "pharma", "pfizer", "moderna", "biotech", "fda", "vaccine", "healthcare"],
#     # 非必需消费品 (Consumer Discretionary)
#     ["consumer", "tesla", "amazon", "alibaba", "retail", "auto", "carmaker"],
#     # 通信服务 (Communication Services)
#     ["telecom", "verizon", "at&t", "disney", "netflix", "google", "meta"],
#     # 原材料 (Materials)
#     ["materials", "mining", "rio tinto", "bhp", "commodity", "metals", "lithium"],
#     # 房地产 (Real Estate)
#     ["real estate", "commercial property", "housing", "reits"]
# ]

In [ ]:
docs = filtered_news['Content'].tolist()
timestamps = filtered_news['Date'].tolist()

# hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model, 
                       # min_topic_size=200,
                    #    nr_topics=20,
                        # seed_topic_list=GICS_SEED_KEYWORDS,
                        verbose=True)
topics, probs = topic_model.fit_transform(docs)
topic_info = topic_model.get_topic_info()
print("Topic Info:")
print(topic_info)

2025-07-02 22:53:48,445 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/131 [00:00<?, ?it/s]

2025-07-02 22:54:33,670 - BERTopic - Embedding - Completed ✓
2025-07-02 22:54:33,671 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-07-02 22:54:34,720 - BERTopic - Dimensionality - Completed ✓
2025-07-02 22:54:34,721 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-07-02 22:54:34,786 - BERTopic - Cluster - Completed ✓
2025-07-02 22:54:34,788 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-07-02 22:54:35,471 - BERTopic - Representation - Completed ✓


Topic Info:
     Topic  Count                                     Name  \
0       -1   1096                         -1_the_of_to_and   
1        0     98                         0_her_was_he_the   
2        1     88               1_estate_space_real_square   
3        2     88        2_security_airport_faa_passengers   
4        3     87  3_taliban_pakistan_afghanistan_alliance   
..     ...    ...                                      ...   
102    101     11                  101_coke_dulce_leche_de   
103    102     11            102_iran_iranian_turkey_blair   
104    103     10                 103_billion_york_tax_new   
105    104     10             104_loss_cents_million_share   
106    105     10             105_or_nasdaq_gained_quarter   

                                        Representation  \
0      [the, of, to, and, in, that, for, it, is, said]   
1    [her, was, he, the, center, she, and, of, his,...   
2    [estate, space, real, square, tenants, office,...   
3    [secur

In [66]:
topic_model.visualize_topics() 

In [12]:
topic_model.topic_embeddings_

array([[-0.00710328, -0.02099075,  0.00372117, ..., -0.09165002,
        -0.01679543,  0.01814516],
       [-0.01499701, -0.0206952 ,  0.01457956, ..., -0.12852328,
        -0.0144714 ,  0.01559433],
       [-0.03804268, -0.04707636,  0.03129139, ..., -0.10243652,
        -0.01138562,  0.01216789],
       ...,
       [ 0.00487667, -0.01531598,  0.03141178, ..., -0.13084823,
        -0.06148916,  0.05292897],
       [ 0.07394592, -0.04786346,  0.03183304, ..., -0.00445615,
        -0.03005093,  0.00427538],
       [ 0.02253365,  0.00237644,  0.03121332, ..., -0.05266342,
        -0.0487191 ,  0.0067972 ]], dtype=float32)

In [95]:
# War Threats (Category 1), Peace Threats (Category 2), Military Buildups (Category 3), Nuclear Threats (Category 4), Terror Threats (Category 5), Beginning of War (Category 6), Escalation of War (Category 7), Terror Acts (Category 8)
GPR_categories = ['War Threats', 'Peace Threats', 'Military Buildups', 'Nuclear Threats', 'Terror Threats', 'Beginning of War', 'Escalation of War', 'Terror Acts']
# GPR_categories = ['Economic Policy & Budget', 'Environment', 'Trade', 'Institutions & Political Process', 'Health', 'Security & Defense', 'Tax Policy', 'Technology & Infrastructure']

# Compute the similarity between the topics and the GPR categories
# Using the topic embeddings from BERTopic
topic_embeddings = topic_model.topic_embeddings_
gpr_embeddings = [embedding_model.encode(category) for category in GPR_categories]

from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(topic_embeddings, gpr_embeddings)
# Show all topics with all GPR categories
similarity_df = pd.DataFrame(similarities, columns=GPR_categories)
similarity_df.index = [f'Topic {i}' for i in range(len(topic_embeddings))]
print("Similarity between topics and GPR categories:")
similarity_df.head(50)  # Show the first 10 topics for brevity

Similarity between topics and GPR categories:


,War Threats,Peace Threats,Military Buildups,Nuclear Threats,Terror Threats,Beginning of War,Escalation of War,Terror Acts
Topic 0,0.407445,0.382943,0.353620,0.373854,0.505942,0.201828,0.314179,0.403943
Topic 1,0.152226,0.137867,0.185044,0.172596,0.303672,0.175619,0.097915,0.312754
Topic 2,0.186158,0.211986,0.198169,0.174005,0.302318,0.080817,0.161023,0.258663
Topic 3,0.279980,0.288713,0.229067,0.295498,0.453540,0.084204,0.194392,0.341209
Topic 4,0.396796,0.453112,0.212338,0.296720,0.406994,0.292533,0.369925,0.341001
Topic 5,0.250987,0.213942,0.239426,0.263099,0.344631,0.178820,0.199245,0.304641
Topic 6,0.257520,0.259276,0.207796,0.240361,0.424922,0.189536,0.187954,0.447911
Topic 7,0.114746,0.035525,0.137593,0.147160,0.079110,0.072344,0.104701,0.065192
Topic 8,0.374131,0.442955,0.239783,0.264649,0.537508,0.185559,0.310001,0.482520
Topic 9,0.182414,0.138148,0.162790,0.172331,0.172594,0.111564,0.127534,0.085865


In [84]:
# get topic count
topic_counts = topic_model.get_topic_info().set_index('Topic')['Count']
# print(int(topic_counts[-1]))

# add topic counts to similarity_df
similarity_df['Topic Count'] = 0
# Iterate through the similarity_df and set the Topic Count
for idx, row in similarity_df.iterrows():
    # print(f"Processing {idx}...")
    idx_num = int(idx.split(' ')[-1]) - 1  # Extract the topic number from the index
    similarity_df.at[idx, 'Topic Count'] = int(topic_counts[idx_num])

similarity_df

,War Threats,Peace Threats,Military Buildups,Nuclear Threats,Terror Threats,Beginning of War,Escalation of War,Terror Acts,Topic Count
Topic 0,0.407445,0.382943,0.353620,0.373854,0.505942,0.201828,0.314179,0.403943,1096
Topic 1,0.152226,0.137867,0.185044,0.172596,0.303672,0.175619,0.097915,0.312754,98
Topic 2,0.186158,0.211986,0.198169,0.174005,0.302318,0.080817,0.161023,0.258663,88
Topic 3,0.279980,0.288713,0.229067,0.295498,0.453540,0.084204,0.194392,0.341209,88
Topic 4,0.396796,0.453112,0.212338,0.296720,0.406994,0.292533,0.369925,0.341001,87
...,...,...,...,...,...,...,...,...,...
Topic 102,0.109937,0.099911,0.115224,0.090374,0.117416,0.124450,0.168963,0.100742,11
Topic 103,0.438931,0.422880,0.356742,0.469906,0.474147,0.239160,0.375693,0.380113,11
Topic 104,0.276555,0.240952,0.411128,0.212350,0.290978,0.158867,0.233859,0.255767,10
Topic 105,0.075789,0.049002,0.122411,0.080879,0.095903,0.049521,0.052969,0.017043,10


In [25]:
similarity_df.to_excel(f'similarity_df_{year}.xlsx', index=True)

In [90]:
# Sort the similarity_df by the first GPR categories
similarity_df_sorted = similarity_df.sort_values(by=GPR_categories[0], ascending=False)
similarity_df_sorted.head(5)

,War Threats,Peace Threats,Military Buildups,Nuclear Threats,Terror Threats,Beginning of War,Escalation of War,Terror Acts,Topic Count
Topic 42,0.544486,0.588386,0.349956,0.483012,0.640226,0.314885,0.430728,0.552801,31
Topic 36,0.497201,0.470271,0.301899,0.461821,0.520639,0.271790,0.323664,0.391296,32
Topic 58,0.442863,0.496717,0.346416,0.310726,0.522256,0.286597,0.381482,0.447683,24
Topic 103,0.438931,0.422880,0.356742,0.469906,0.474147,0.239160,0.375693,0.380113,11
Topic 34,0.433330,0.273969,0.565565,0.376334,0.314819,0.273539,0.394283,0.278278,33


In [75]:
def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models """
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):

        # Extract top 5 words per topic per model
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:2])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:2])
        df.loc[len(df)] = [topic, og_words, new_words]

    return df

In [76]:
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
"""

# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY")

representation_model = OpenAI(
    client, model='gpt-4o' ,exponential_backoff=True, chat=True, prompt=prompt
)

topic_model.update_topics(docs, representation_model=representation_model)

# Show topic differences
# topic_differences(topic_model, original_topics)


100%|██████████| 107/107 [03:42<00:00,  2.08s/it]


In [77]:
df_result = topic_model.get_topic_info()
# drop Representative_Docs
df_result = df_result.drop(columns=['Representative_Docs'], errors='ignore')
# Save the result to a CSV file
df_result.to_csv(f'./result_{year}_v1.csv')
df_result

,Topic,Count,Name,Representation
0,-1,1096,-1_Post-9/11 Economic Impact and Security Meas...,[Post-9/11 Economic Impact and Security Measures]
1,0,98,0_September 11 World Trade Center Attack Survi...,[September 11 World Trade Center Attack Surviv...
2,1,88,1_Post-9/11 Manhattan Real Estate Impact and A...,[Post-9/11 Manhattan Real Estate Impact and Ad...
3,2,88,2_Post-9/11 Airline Security Measures,[Post-9/11 Airline Security Measures]
4,3,87,3_Afghanistan-Pakistan Relations and the Talib...,[Afghanistan-Pakistan Relations and the Taliba...
...,...,...,...,...
102,101,11,101_Coca-Cola's Advertising Consolidation and ...,[Coca-Cola's Advertising Consolidation and Dul...
103,102,11,102_U.S.-Iran Relations and Counterterrorism E...,[U.S.-Iran Relations and Counterterrorism Effo...
104,103,10,103_Post-9/11 Economic Aid and Infrastructure ...,[Post-9/11 Economic Aid and Infrastructure Reb...
105,104,10,104_Financial Performance and Market Reaction ...,[Financial Performance and Market Reaction of ...


In [26]:
for year in range(2008, 2025):
    print(f"Processing year: {year}")
    
    df_gpr = pd.read_csv(f'./Result/GPR_Peaks/data/gpr_peaks_{year}.csv')
    filtered_news = df_news[df_news['Date'].isin(df_gpr['date'])]
    docs = filtered_news['Content'].tolist()
    timestamps = filtered_news['Date'].tolist()

    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    topic_model = BERTopic(embedding_model=embedding_model, 
                           # min_topic_size=200,
                           nr_topics=5,
                           verbose=True)
    topics, probs = topic_model.fit_transform(docs)
    topic_info = topic_model.get_topic_info()

    original_topics = deepcopy(topic_model.topic_representations_)

    prompt = """
    I have a topic that contains the following documents:
    [DOCUMENTS]

    The topic is described by the following keywords: [KEYWORDS]

    Based on the information above, extract a short topic label in the following format:
    topic: <short topic label>
    """

    # Update our topic representations using GPT-3.5
    client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY")

    representation_model = OpenAI(
        client, model='gpt-4o' ,exponential_backoff=True, chat=True, prompt=prompt
    )

    topic_model.update_topics(docs, representation_model=representation_model)

    df_result = topic_model.get_topic_info()
    # drop Representative_Docs
    df_result = df_result.drop(columns=['Representative_Docs'], errors='ignore')
    # Save the result to a CSV file
    df_result.to_csv(f'./result_{year}.csv')

Processing year: 2008


2025-06-30 20:18:28,970 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 112/112 [00:21<00:00,  5.30it/s]
2025-06-30 20:18:50,388 - BERTopic - Embedding - Completed ✓
2025-06-30 20:18:50,388 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-06-30 20:18:57,753 - BERTopic - Dimensionality - Completed ✓
2025-06-30 20:18:57,754 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-06-30 20:18:57,818 - BERTopic - Cluster - Completed ✓
2025-06-30 20:18:57,819 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-06-30 20:18:58,486 - BERTopic - Representation - Completed ✓
2025-06-30 20:18:58,487 - BERTopic - Topic reduction - Reducing number of topics
2025-06-30 20:18:58,490 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-06-30 20:18:59,090 - BERTopic - Representation - Completed ✓
2025-06-30 20:18:59,092 - BERTopic - Topic reduction - Re

Processing year: 2009


2025-06-30 20:19:07,832 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 156/156 [00:27<00:00,  5.61it/s]
2025-06-30 20:19:36,023 - BERTopic - Embedding - Completed ✓
2025-06-30 20:19:36,024 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-06-30 20:19:37,575 - BERTopic - Dimensionality - Completed ✓
2025-06-30 20:19:37,575 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-06-30 20:19:37,667 - BERTopic - Cluster - Completed ✓
2025-06-30 20:19:37,668 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-06-30 20:19:38,510 - BERTopic - Representation - Completed ✓
2025-06-30 20:19:38,511 - BERTopic - Topic reduction - Reducing number of topics
2025-06-30 20:19:38,515 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-06-30 20:19:39,331 - BERTopic - Representation - Completed ✓
2025-06-30 20:19:39,332 - BERTopic - Topic reduction - Re

Processing year: 2010


...

2025-06-30 20:27:54,515 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 81/81 [00:14<00:00,  5.67it/s]
2025-06-30 20:28:09,013 - BERTopic - Embedding - Completed ✓
2025-06-30 20:28:09,013 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-06-30 20:28:12,965 - BERTopic - Dimensionality - Completed ✓
2025-06-30 20:28:12,965 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-06-30 20:28:13,004 - BERTopic - Cluster - Completed ✓
2025-06-30 20:28:13,005 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-06-30 20:28:13,553 - BERTopic - Representation - Completed ✓
2025-06-30 20:28:13,554 - BERTopic - Topic reduction - Reducing number of topics
2025-06-30 20:28:13,557 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-06-30 20:28:14,099 - BERTopic - Representation - Completed ✓
2025-06-30 20:28:14,100 - BERTopic - Topic reduction - Redu

Processing year: 2023


2025-06-30 20:28:22,967 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 96/96 [00:14<00:00,  6.49it/s]
2025-06-30 20:28:37,844 - BERTopic - Embedding - Completed ✓
2025-06-30 20:28:37,845 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-06-30 20:28:43,154 - BERTopic - Dimensionality - Completed ✓
2025-06-30 20:28:43,154 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-06-30 20:28:43,203 - BERTopic - Cluster - Completed ✓
2025-06-30 20:28:43,203 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-06-30 20:28:43,417 - BERTopic - Representation - Completed ✓
2025-06-30 20:28:43,418 - BERTopic - Topic reduction - Reducing number of topics
2025-06-30 20:28:43,422 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-06-30 20:28:43,615 - BERTopic - Representation - Completed ✓
2025-06-30 20:28:43,616 - BERTopic - Topic reduction - Redu

Processing year: 2024


2025-06-30 20:28:52,801 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 46/46 [00:07<00:00,  6.26it/s]
2025-06-30 20:29:00,161 - BERTopic - Embedding - Completed ✓
2025-06-30 20:29:00,162 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-06-30 20:29:01,784 - BERTopic - Dimensionality - Completed ✓
2025-06-30 20:29:01,784 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-06-30 20:29:01,807 - BERTopic - Cluster - Completed ✓
2025-06-30 20:29:01,807 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-06-30 20:29:01,833 - BERTopic - Representation - Completed ✓
2025-06-30 20:29:01,833 - BERTopic - Topic reduction - Reducing number of topics
2025-06-30 20:29:01,834 - BERTopic - Topic reduction - Number of topics (5) is equal or higher than the clustered topics(4).
2025-06-30 20:29:01,834 - BERTopic - Representation - Fine-tuning topics using representation models.
